Forecasting UK Energy Prices 

In this short project, I aim to use the programming skills I have learnt during my time at university, as well as my own interest in using data to derive insights about the real world problems, to attempt to fit a statistical model to real UK energy data, in hopes to try and make predicitons about the settlement price at the next time period.

Part 1: Retrieving Data about Energy Price:
The data I have used in this project has come from Elexon BMRS: a source of energy data for UK markets. Specifically I collected data starting from the January 1st 2026 to September 23rd 2026 (note only partial data was available on the 23rd September) from the Market Index Price dataset.

In [1]:
import pathlib
import pandas as pd
import datetime as dt
import time

def generate_data():
    d = dt.date(2026, 1, 1)
    o = dt.date(2026,1,1)
    t=dt.date.today()
    delta = t - o

    index = (delta.days//6) + 1

    for i in range((index)):    # Only used 45 becasue this is when I last collected data
        url = f"https://data.elexon.co.uk/bmrs/api/v1/datasets/MID?from={d}T00%3A00Z&to={d+dt.timedelta(days=5)}T23%3A30Z&dataProviders=APXMIDP&format=csv"
        df = pd.read_csv(url)
        df.to_csv(f"test_{i}.csv", sep=",")
        d += dt.timedelta(days=6)
        time.sleep(0.05)


    df_main = pd.concat([pd.read_csv(f"test_{i}.csv") for i in range((index))]).sort_values(by=["SettlementDate","SettlementPeriod"], ignore_index=True)
    df_main.to_csv("check", sep=",")

    for i in range(index):
        pathlib.Path.unlink(f"test_{i}.csv")
        time.sleep(0.05)

    return df_main

df = generate_data()
df = df.drop(columns=["Unnamed: 0", "DataProvider","Dataset"])
df["StartTime"] = pd.to_datetime(df["StartTime"]).dt.tz_convert(None)
df["SettlementDate"] = pd.to_datetime(df["SettlementDate"])
df.set_index(df["StartTime"], inplace=True)
df = df.drop(columns=["StartTime"])

Before starting to build a model to predict settlement price, I firstly performed EDA an fit a baseline model (Naive Forecast) to (a) understand underlying trends in the data and (b) have some initial metrics to compare my models performance to.

In [ ]:
def mean_sp():    # The settlement period calculations will compute the statistics up to thee latest available settlement period
    mean_settlement_vals = []
    for i in range(1,49):
        mean_settlement_vals.append(df[df["SettlementPeriod"]==i]["Price"].mean())

    return mean_settlement_vals

def sd_sp():
    sd_settlement_vals = []
    for i in range(1,49):
        sd_settlement_vals.append(df[df["SettlementPeriod"]==i]["Price"].std())

    return sd_settlement_vals

def mean_daily():    # The daily calculations will calculate the daily statistics up to the day the program is run
    mean_daily_vals = []
    for i in range(7):
        mean_daily_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].mean())
        
    return mean_daily_vals

def sd_daily():
    sd_daily_vals = []
    for i in range(7):
        sd_daily_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].std())
        
    return sd_daily_vals

def mean_monthly():    # The monthly calculations will calculate the monthly statisics up to the day of the month the program is run
    mean_monthly_vals =[]
    for i in range(1,(dt.date.today().month)+1):
        mean_monthly_vals.append(df[df["SettlementDate"].dt.month==i]["Price"].mean())

    return mean_monthly_vals

def sd_monthly():
    sd_monthly_vals =[]
    for i in range(1,(dt.date.today().month)+1):
        sd_monthly_vals.append(df[df["SettlementDate"].dt.month==i]["Price"].std())

    return sd_monthly_vals


# The next function computes the coefficient of variation for each day of the week

def coef_of_var():
    cov_vals = []
    for i in range(7):
        cov_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].mean()/df[df["SettlementDate"].dt.day_of_week==i]["Price"].std())

    return cov_vals

mean_monthly()

[np.float64(97.08569220430108),
 np.float64(79.72023065476189),
 np.float64(95.53942126514131),
 np.float64(82.21836805555556),
 np.float64(104.74385752688171),
 np.float64(96.19275),
 np.float64(105.78786290322579),
 np.float64(129.86952956989245),
 np.float64(129.5878731074261)]

The below code visualises each of the above situations to give an idea of the shape of the data

In [3]:
# import matplotlib.pyplot as plt

# prompt = ("Enter the name of the function you want to plot ")
# prompt += "\n(For instance, if you would like to see the mean daily price, enter 'mean_daily()')"

# data_to_collect = input(prompt)

# fig, ax = plt.subplots()

# if 